# Fatigue & Burnout Detection — Google Colab Master Prompt
## Temporal Fatigue Modelling via Arousal-Based Pseudo-Labelling

### Step 1 & 2 — Environment Setup & Path Configuration

In [1]:
import os
import random
import copy
import shutil
import pickle
import glob
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.stats import linregress

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, f1_score
import warnings

warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Running in Google Colab environment.")

# Paths Configuration based on Drive Layout
base_dir = '/content/drive/MyDrive/Fatigue_Burnout_detection'
raw_data_dir = os.path.join(base_dir, 'data', 'raw')

# Create necessary directories
dirs_to_create = [
    os.path.join(base_dir, "features", "npy"),
    os.path.join(base_dir, "features", "csv"),
    os.path.join(base_dir, "models", "checkpoints"),
    os.path.join(base_dir, "models", "final"),
    os.path.join(base_dir, "results", "training_curves"),
    os.path.join(base_dir, "results", "metrics")
]

for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)

print(f"Base Directory set to: {base_dir}")
print(f"Raw Data Directory set to: {raw_data_dir}")

# Set SEED globally for reproducibility
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cpu':
    print("WARNING: No GPU available! Training will be slow.")



Mounted at /content/drive
Running in Google Colab environment.
Base Directory set to: /content/drive/MyDrive/Fatigue_Burnout_detection
Raw Data Directory set to: /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw
Using device: cuda


### Step 3 — Load Pre-trained Emotion Model

In [2]:
# Define Stage 1 Emotion Model Architecture (CRNN)
class CRNN(nn.Module):
    def __init__(self, n_mels: int = 80, n_classes: int = 3, hidden_size: int = 192, dropout: float = 0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(64, 96, kernel_size=3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
        )
        self.gru = nn.GRU(input_size=(n_mels // 8) * 96, hidden_size=hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)
        x = self.cnn(x)
        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2).contiguous().view(b, t, c * f)
        out, _ = self.gru(x)
        out = out[:, -1, :]
        return self.head(out)

# Update to point to your local pretrained emotion model!
emotion_model_path = '/content/drive/MyDrive/VoiceAnalysis DL+GRU/crnn_best.pt'

try:
    emotion_model = CRNN(n_classes=3).to(device)
    emotion_model.load_state_dict(torch.load(emotion_model_path, map_location=device))
    emotion_model.eval()
    for param in emotion_model.parameters():
        param.requires_grad = False
    print("CRNN Model successfully loaded!")
except Exception as e:
    print(f"Error loading model: {e}")

def get_emotion_probs(audio_path, model, device):
    y, sr = librosa.load(audio_path, sr=16000, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=512, n_mels=80)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    input_tensor = torch.tensor(mel_spec_db, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    return probs[0], probs[1], probs[2]



CRNN Model successfully loaded!


### Step 4 & 5 — Acoustic Feature Extraction & Arousal Computation

In [3]:
def extract_acoustic_features(audio_path):
    y, sr = librosa.load(audio_path, sr=16000, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)

    rms_mean = np.mean(librosa.feature.rms(y=y)[0])

    f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=75, fmax=300, sr=sr)
    f0_voiced = f0[~np.isnan(f0)]
    f0_mean = np.mean(f0_voiced) if len(f0_voiced) > 0 else 0
    f0_var = np.var(f0_voiced) if len(f0_voiced) > 0 else 0

    centroid_mean = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)[0])
    rolloff_mean = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)[0])
    zcr_mean = np.mean(librosa.feature.zero_crossing_rate(y=y)[0])

    return [rms_mean, f0_mean, f0_var, centroid_mean, rolloff_mean, zcr_mean]

all_audio_paths = glob.glob(os.path.join(raw_data_dir, "**", "*.wav"), recursive=True)
if len(all_audio_paths) == 0:
    raise ValueError(f"No .wav files found in {raw_data_dir}. Check your directory path!")

raw_features = []
emotion_probs_list = []
metadata_list = []

for path in tqdm(all_audio_paths, desc="Processing audio files"):
    filename = os.path.basename(path)
    speaker_id = "unknown"
    dataset = "Unknown"

    if "CREMA" in path or filename[0].isdigit():
        speaker_id = filename.split('_')[0]
        dataset = "CREMA-D"
    elif "Actor" in path or filename.startswith("03-"):
        parts = filename.split('-')
        if len(parts) >= 7:
            speaker_id = "Actor_" + parts[-1].split('.')[0]
        elif "Actor" in path:
            speaker_id = "Actor_" + filename.split('_')[1].split('.')[0]
        dataset = "RAVDESS"
    elif "OAF" in filename or "YAF" in filename:
        speaker_id = "OAF" if "OAF" in filename else "YAF"
        dataset = "TESS"

    feats = extract_acoustic_features(path)
    p_pos, p_neg, p_neu = get_emotion_probs(path, emotion_model, device)

    raw_features.append(feats)
    emotion_probs_list.append([p_pos, p_neg, p_neu])
    metadata_list.append({"file": filename, "speaker_id": speaker_id, "dataset": dataset})

raw_acoustic = np.array(raw_features)
emotion_probs = np.array(emotion_probs_list)
metadata_df = pd.DataFrame(metadata_list)

# Global Scaler (FOR PSEUDO-LABELLING & CSV PURPOSES ONLY - NOT USED IN CROSS-VALIDATION)
# To prevent data leakage, training uses a fold-specific scaler later.
global_scaler = MinMaxScaler()
raw_acoustic_norm = global_scaler.fit_transform(raw_acoustic)

npy_dir = os.path.join(base_dir, 'features', 'npy')
csv_dir = os.path.join(base_dir, 'features', 'csv')

# Note: We do NOT save global_scaler to pkl as the definitive model to avoid leakage during inference.
# We will save the scaler trained on Fold 1 later.

A_acoustic = (0.30 * raw_acoustic_norm[:, 0] + 0.25 * raw_acoustic_norm[:, 1] +
              0.15 * raw_acoustic_norm[:, 2] + 0.15 * raw_acoustic_norm[:, 5] +
              0.10 * raw_acoustic_norm[:, 3] + 0.05 * raw_acoustic_norm[:, 4])

A_emotion = 0.8 * emotion_probs[:, 0] + 0.2 * emotion_probs[:, 1] + 0.4 * emotion_probs[:, 2]
alpha = 0.60
A_final = alpha * A_emotion + (1 - alpha) * A_acoustic

beta = 0.70
F_score = beta * (1 - A_final) + (1 - beta) * emotion_probs[:, 1]

kmeans = KMeans(n_clusters=3, random_state=SEED, n_init=10)
labels_raw = kmeans.fit_predict(F_score.reshape(-1, 1))

order = np.argsort(kmeans.cluster_centers_.sum(axis=1))
mapping = {order[0]: 0, order[1]: 1, order[2]: 2}
mapping_str = {0: "Non-Fatigued", 1: "Mild Fatigue", 2: "High Fatigue / Burnout"}
pseudo_labels = np.array([mapping[l] for l in labels_raw])
pseudo_label_str = [mapping_str[l] for l in pseudo_labels]

with open(os.path.join(base_dir, "results", "metrics", "pseudo_label_stats.txt"), "w") as f:
    f.write(f"Centroids: {kmeans.cluster_centers_.flatten()}\n")
    f.write(f"Sorted Object Mapping: {mapping} (0: Non-Fatigued, 1: Mild, 2: High Burnout)\n")

with open(os.path.join(npy_dir, "kmeans_model.pkl"), "wb") as f:
    pickle.dump(kmeans, f)
np.save(os.path.join(npy_dir, "kmeans_thresholds.npy"), kmeans.cluster_centers_)

# Plot 1: F_score Distribution
plt.figure(figsize=(10, 6))
plt.hist(F_score, bins=50, color='gray', alpha=0.6, edgecolor='black')
centroids = np.sort(kmeans.cluster_centers_.flatten())
b1 = (centroids[0] + centroids[1]) / 2
b2 = (centroids[1] + centroids[2]) / 2
plt.axvline(b1, color='red', linestyle='--')
plt.axvline(b2, color='red', linestyle='--')
plt.axvspan(0, b1, alpha=0.2, color='green', label='Non-Fatigued')
plt.axvspan(b1, b2, alpha=0.2, color='orange', label='Mild Fatigue')
plt.axvspan(b2, 1.0, alpha=0.2, color='red', label='High Fatigue')
plt.title('F_score Distribution with K-Means Clusters')
plt.xlabel('F_score')
plt.ylabel('Count')
plt.legend()
plt.savefig(os.path.join(base_dir, "results", "metrics", "fscore_distribution.png"))
plt.close()

# Assemble raw dataframe features
features_df = pd.DataFrame(raw_acoustic, columns=['RMS_mean', 'F0_mean', 'F0_var', 'Centroid_mean', 'Rolloff_mean', 'ZCR_mean'])
features_df['P_pos'] = emotion_probs[:, 0]
features_df['P_neg'] = emotion_probs[:, 1]
features_df['P_neu'] = emotion_probs[:, 2]
features_df['A_acoustic'] = A_acoustic
features_df['A_emotion'] = A_emotion
features_df['A_final'] = A_final
features_df['F_score'] = F_score
features_df['Pseudo_Label'] = pseudo_labels
features_df['pseudo_label_str'] = pseudo_label_str
features_df = pd.concat([metadata_df, features_df], axis=1)

# Plot 2: Label Distribution Grouped Chart
plt.figure(figsize=(10, 6))
counts = features_df.groupby(['dataset', 'pseudo_label_str']).size().unstack(fill_value=0)
col_order = [mapping_str[0], mapping_str[1], mapping_str[2]]
counts = counts.reindex(columns=[c for c in col_order if c in counts.columns])
ax = counts.plot(kind='bar', figsize=(10,6), colormap='viridis')
for container in ax.containers:
    ax.bar_label(container)
plt.title('Pseudo-Label Distribution by Dataset')
plt.xlabel('Dataset')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Fatigue Class')
plt.tight_layout()
plt.savefig(os.path.join(base_dir, "results", "metrics", "label_distribution.png"))
plt.close()



Processing audio files: 100%|██████████| 11682/11682 [1:09:15<00:00,  2.81it/s]


<Figure size 1000x600 with 0 Axes>

In [4]:
# Check how many files got labelled Unknown
print(features_df['dataset'].value_counts())

# Preview a few RAVDESS-like filenames
ravdess_candidates = [p for p in all_audio_paths if 'ravdess' in p.lower() or 'actor' in p.lower()]
print(ravdess_candidates[:5])

dataset
CREMA-D    8882
TESS       2800
Name: count, dtype: int64
['/content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/RAVDESS/Actor_01/03-01-01-01-02-01-01.wav', '/content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/RAVDESS/Actor_01/03-01-01-01-01-01-01.wav', '/content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/RAVDESS/Actor_01/03-01-02-02-01-01-01.wav', '/content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/RAVDESS/Actor_01/03-01-02-01-02-01-01.wav', '/content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/RAVDESS/Actor_01/03-01-01-01-02-02-01.wav']


In [5]:
# ── RAVDESS patch ── no need to re-run feature extraction ──────────────────

ravdess_raw_features = []
ravdess_emotion_probs = []
ravdess_metadata = []

ravdess_paths = [p for p in all_audio_paths if 'ravdess' in p.lower() or 'Actor' in p]

for path in tqdm(ravdess_paths, desc="Processing RAVDESS files"):
    filename = os.path.basename(path)

    # RAVDESS filename format: 03-01-01-01-01-01-01.wav
    # Actor ID is in the parent folder name: Actor_01 → speaker "Actor_01"
    parent_folder = os.path.basename(os.path.dirname(path))  # e.g. "Actor_01"
    speaker_id = parent_folder  # clean, reliable
    dataset = "RAVDESS"

    try:
        feats = extract_acoustic_features(path)
        p_pos, p_neg, p_neu = get_emotion_probs(path, emotion_model, device)

        ravdess_raw_features.append(feats)
        ravdess_emotion_probs.append([p_pos, p_neg, p_neu])
        ravdess_metadata.append({
            "file": filename,
            "speaker_id": speaker_id,
            "dataset": dataset
        })
    except Exception as e:
        print(f"Skipped {filename}: {e}")

print(f"\nRAVDESS files processed: {len(ravdess_metadata)}")

Processing RAVDESS files: 100%|██████████| 1440/1440 [03:48<00:00,  6.31it/s]


RAVDESS files processed: 1440


In [6]:
# Build RAVDESS dataframes
ravdess_raw_ac = np.array(ravdess_raw_features)
ravdess_emo = np.array(ravdess_emotion_probs)
ravdess_meta_df = pd.DataFrame(ravdess_metadata)

# Normalise using global scaler (pseudo-label purposes only)
ravdess_ac_norm = global_scaler.transform(ravdess_raw_ac)

# Compute scores
A_ac_r = (0.30*ravdess_ac_norm[:,0] + 0.25*ravdess_ac_norm[:,1] +
           0.15*ravdess_ac_norm[:,2] + 0.15*ravdess_ac_norm[:,5] +
           0.10*ravdess_ac_norm[:,3] + 0.05*ravdess_ac_norm[:,4])
A_em_r = 0.8*ravdess_emo[:,0] + 0.2*ravdess_emo[:,1] + 0.4*ravdess_emo[:,2]
A_fin_r = alpha * A_em_r + (1 - alpha) * A_ac_r
F_sc_r  = beta * (1 - A_fin_r) + (1 - beta) * ravdess_emo[:,1]

# Assign pseudo-labels using the EXISTING kmeans (do not refit)
labels_raw_r = kmeans.predict(F_sc_r.reshape(-1, 1))
pseudo_labels_r = np.array([mapping[l] for l in labels_raw_r])
pseudo_label_str_r = [mapping_str[l] for l in pseudo_labels_r]

# Build RAVDESS features dataframe
ravdess_feat_df = pd.DataFrame(ravdess_raw_ac,
    columns=['RMS_mean','F0_mean','F0_var','Centroid_mean','Rolloff_mean','ZCR_mean'])
ravdess_feat_df['P_pos']           = ravdess_emo[:,0]
ravdess_feat_df['P_neg']           = ravdess_emo[:,1]
ravdess_feat_df['P_neu']           = ravdess_emo[:,2]
ravdess_feat_df['A_acoustic']      = A_ac_r
ravdess_feat_df['A_emotion']       = A_em_r
ravdess_feat_df['A_final']         = A_fin_r
ravdess_feat_df['F_score']         = F_sc_r
ravdess_feat_df['Pseudo_Label']    = pseudo_labels_r
ravdess_feat_df['pseudo_label_str']= pseudo_label_str_r
ravdess_feat_df = pd.concat([ravdess_meta_df.reset_index(drop=True),
                              ravdess_feat_df.reset_index(drop=True)], axis=1)

# Merge into main features_df
features_df = pd.concat([features_df, ravdess_feat_df], ignore_index=True)

# Also extend the numpy arrays used for temporal window construction
raw_acoustic      = np.vstack([raw_acoustic, ravdess_raw_ac])
emotion_probs     = np.vstack([emotion_probs, ravdess_emo])
A_acoustic        = np.concatenate([A_acoustic, A_ac_r])
A_emotion         = np.concatenate([A_emotion, A_em_r])
A_final           = np.concatenate([A_final, A_fin_r])
F_score           = np.concatenate([F_score, F_sc_r])
pseudo_labels     = np.concatenate([pseudo_labels, pseudo_labels_r])

print(f"\nUpdated dataset counts:")
print(features_df['dataset'].value_counts())


Updated dataset counts:
dataset
CREMA-D    8882
TESS       2800
RAVDESS    1440
Name: count, dtype: int64


In [7]:
# Re-save sample_features CSV
chunk_size = 3000
num_chunks = len(features_df) // chunk_size + (1 if len(features_df) % chunk_size > 0 else 0)
for i in range(num_chunks):
    features_df.iloc[i*chunk_size:(i+1)*chunk_size].to_csv(
        os.path.join(csv_dir, f"sample_features_part_{i+1}.csv"), index=False)

# Re-plot label distribution
fig, ax = plt.subplots(figsize=(10, 6))
counts = features_df.groupby(['dataset', 'pseudo_label_str']).size().unstack(fill_value=0)
col_order = [mapping_str[0], mapping_str[1], mapping_str[2]]
counts = counts.reindex(columns=[c for c in col_order if c in counts.columns])
counts.plot(kind='bar', ax=ax, colormap='viridis')
for container in ax.containers:
    ax.bar_label(container)
ax.set_title('Pseudo-Label Distribution by Dataset')
ax.set_xlabel('Dataset')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Fatigue Class')
plt.tight_layout()
plt.savefig(os.path.join(base_dir, "results", "metrics", "label_distribution.png"))
plt.close()
print("Done. label_distribution.png updated with all 3 datasets.")

Done. label_distribution.png updated with all 3 datasets.


### Step 6 & 7 & 8 — Temporal Window Construction (Per-Speaker) & Exports

In [10]:
W = 10
S = 5
gamma = 0.30

window_features = []

# We build raw acoustic and emotion windows natively to compute them fold-by-fold properly later
raw_sequences_list = []
emotion_seqs_list = []
temporal_labels = []

# Global sequences placeholder (fully scaled via global scaler) just to give you the reference shapes
global_scaled_sequences = []

for speaker in features_df['speaker_id'].unique():
    spk_df = features_df[features_df['speaker_id'] == speaker].sort_values('file')
    if len(spk_df) < W:
        continue

    for i in range(0, len(spk_df) - W + 1, S):
        window = spk_df.iloc[i:i+W]

        A_fin_seq = window['A_final'].values
        F_score_seq = window['F_score'].values

        slope, _, _, _, _ = linregress(range(W), A_fin_seq)
        if np.isnan(slope): slope = 0

        F_adj = np.mean(F_score_seq) + gamma * max(0, -slope)
        win_label = int(window['Pseudo_Label'].values.max())

        # Raw Data Arrays
        raw_ac = window[['RMS_mean', 'F0_mean', 'F0_var', 'Centroid_mean', 'Rolloff_mean', 'ZCR_mean']].values
        emo_arr = window[['P_pos', 'P_neg', 'P_neu']].values
        raw_sequences_list.append(raw_ac)
        emotion_seqs_list.append(emo_arr)
        temporal_labels.append(win_label)

        # Globally Scaled Data 3D Arrays (Saved in CSV & NPY)
        A_em_arr = window['A_emotion'].values.reshape(-1, 1)
        A_fin_arr = A_fin_seq.reshape(-1, 1)
        F_sc_arr = F_score_seq.reshape(-1, 1)
        slope_arr = np.full((W, 1), slope)

        # Shape of globally scaled input is (W, 10) natively.
        # This is strictly the normalized form for Global saving (N, 10, 10)
        # However, for training we MUST compute these within the k-fold loop with scaler_fold!
        scaled_ac = global_scaler.transform(raw_ac)
        seq_3d = np.concatenate([scaled_ac, A_em_arr, A_fin_arr, F_sc_arr, slope_arr], axis=1)
        global_scaled_sequences.append(seq_3d)

        # Export dict
        window_features.append({
            'speaker_id': speaker,
            'dataset': window.iloc[0]['dataset'],
            'start_file': window.iloc[0]['file'],
            'end_file': window.iloc[-1]['file'],
            'RMS_mean': raw_ac[:, 0].mean(),
            'F0_mean': raw_ac[:, 1].mean(),
            'F0_var': raw_ac[:, 2].mean(),
            'Centroid_mean': raw_ac[:, 3].mean(),
            'Rolloff_mean': raw_ac[:, 4].mean(),
            'ZCR_mean': raw_ac[:, 5].mean(),
            'A_emotion_mean': A_em_arr.mean(),
            'slope': slope,
            'F_adj': F_adj,
            'Window_Label': win_label,
            'window_label_str': mapping_str[win_label]
        })

# Global variables ready for cross-validation structure
X_raw_np = np.array(raw_sequences_list)
X_emo_np = np.array(emotion_seqs_list)
y_np = np.array(temporal_labels)

# Save Globally Scaled to X_sequences.npy
if len(global_scaled_sequences) > 0:
    X_sequences_global = np.array(global_scaled_sequences)
    np.save(os.path.join(npy_dir, "X_sequences.npy"), X_sequences_global)
    np.save(os.path.join(npy_dir, "y_labels.npy"), y_np)

# Chunked CSV saving logic
chunk_size = 3000

# Sample Features Csv
num_chunks = len(features_df) // chunk_size + (1 if len(features_df) % chunk_size > 0 else 0)
for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(features_df))
    chunk_df = features_df.iloc[start_idx:end_idx]
    chunk_df.to_csv(os.path.join(csv_dir, f"sample_features_part_{i+1}.csv"), index=False)

# Window Features Csv
window_df = pd.DataFrame(window_features)
num_chunks = len(window_df) // chunk_size + (1 if len(window_df) % chunk_size > 0 else 0)
for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(window_df))
    chunk_df = window_df.iloc[start_idx:end_idx]
    chunk_df.to_csv(os.path.join(csv_dir, f"window_features_part_{i+1}.csv"), index=False)

# Label Distribution Csv
label_dist = features_df['Pseudo_Label'].value_counts()
label_dist.to_csv(os.path.join(csv_dir, "label_distribution.csv"))

print(f"Features computed. X_raw shape: {X_raw_np.shape}. y shape: {y_np.shape}")



Features computed. X_raw shape: (2184, 10, 6). y shape: (2184,)


In [11]:
print("Window label distribution:")
print(pd.Series(y_np).value_counts().sort_index())

print("\nWindows per dataset:")
window_df_temp = pd.DataFrame(window_features)
print(window_df_temp['dataset'].value_counts())

print("\nUnique speakers that contributed windows:")
print(window_df_temp['dataset'].value_counts())
print(f"Total windows: {len(y_np)}")

Window label distribution:
0     566
1    1264
2     354
Name: count, dtype: int64

Windows per dataset:
dataset
CREMA-D    1362
TESS        558
RAVDESS     264
Name: count, dtype: int64

Unique speakers that contributed windows:
dataset
CREMA-D    1362
TESS        558
RAVDESS     264
Name: count, dtype: int64
Total windows: 2184


### Step 9 — BiGRU with Attention Architecture

In [12]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(ScaledDotProductAttention, self).__init__()
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.scale = np.sqrt(hidden_dim)

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        attn_weights = F.softmax(scores, dim=-1)

        context = torch.bmm(attn_weights, V)
        context = torch.sum(context, dim=1)
        return context

# Input Dim is strictly 10 per the structure of X_sequences
class BiGRUAttention(nn.Module):
    def __init__(self, input_dim=10, num_classes=3):
        super(BiGRUAttention, self).__init__()

        self.gru1 = nn.GRU(input_size=input_dim, hidden_size=128, bidirectional=True, batch_first=True, dropout=0.3)
        self.gru2 = nn.GRU(input_size=256, hidden_size=64, bidirectional=True, batch_first=True, dropout=0.3)
        self.attention = ScaledDotProductAttention(hidden_dim=128)

        self.fc1 = nn.Linear(128, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.out = nn.Linear(64, num_classes)

    def forward(self, x):
        # We REMOVED unsqueeze(1) because x is strictly 3D shape (batch, W, 10).
        x, _ = self.gru1(x)
        x, _ = self.gru2(x)

        x = self.attention(x)

        x = F.relu(self.fc1(x))
        x = self.bn1(x)
        x = self.dropout1(x)

        x = F.relu(self.fc2(x))
        x = self.dropout2(x)

        return self.out(x)



### Step 10 — 5-Fold Stratified CV Training (Loss-based Early Stopping, Dynamic Sequences)

In [13]:
# Helper Function to construct X per Fold
def build_X_split(ac_scaled, emo):
    # Shape of ac_scaled: (B, W, 6)
    # Shape of emo: (B, W, 3)
    A_ac = 0.3*ac_scaled[:,:,0] + 0.25*ac_scaled[:,:,1] + 0.15*ac_scaled[:,:,2] + 0.15*ac_scaled[:,:,5] + 0.10*ac_scaled[:,:,3] + 0.05*ac_scaled[:,:,4]
    A_em = 0.8*emo[:,:,0] + 0.2*emo[:,:,1] + 0.4*emo[:,:,2]
    A_fin = alpha * A_em + (1 - alpha) * A_ac
    F_sc = beta * (1 - A_fin) + (1 - beta) * emo[:,:,1]

    out = np.zeros((ac_scaled.shape[0], W, 10))
    out[:,:,0:6] = ac_scaled
    out[:,:,6] = A_em
    out[:,:,7] = A_fin
    out[:,:,8] = F_sc

    for b in range(ac_scaled.shape[0]):
        s, _, _, _, _ = linregress(range(W), A_fin[b])
        out[b, :, 9] = s if not np.isnan(s) else 0

    return torch.tensor(out, dtype=torch.float32)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

epochs = 100
batch_size = 64
patience = 10

fold_f1_scores = []
fold_val_preds_probs = {}
fold_val_labels = {}
cv_records = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw_np, y_np)):
    print(f"\n--- Fold {fold+1} Logging ---")

    # Strictly isolate data based on indices
    train_raw = X_raw_np[train_idx]
    val_raw = X_raw_np[val_idx]
    train_emo = X_emo_np[train_idx]
    val_emo = X_emo_np[val_idx]

    # 2. Fit Scaler ONLY on Training Fold Acoustic data to prevent data leakage mapping
    scaler_fold = MinMaxScaler()
    scaler_fold.fit(train_raw.reshape(-1, 6))

    # Save the reference scaler from fold 1
    if fold == 0:
        with open(os.path.join(npy_dir, "feature_scaler.pkl"), "wb") as f:
            pickle.dump(scaler_fold, f)

    # Apply scaled transformations
    train_scaled = scaler_fold.transform(train_raw.reshape(-1, 6)).reshape(train_raw.shape)
    val_scaled = scaler_fold.transform(val_raw.reshape(-1, 6)).reshape(val_raw.shape)

    # Reconstruct robust 3D sequences using fold-local scaling
    X_train_tsr = build_X_split(train_scaled, train_emo)
    X_val_tsr = build_X_split(val_scaled, val_emo)

    y_train_tsr = torch.tensor(y_np[train_idx], dtype=torch.long)
    y_val_tsr = torch.tensor(y_np[val_idx], dtype=torch.long)

    # Class weights per fold iteration (Missing Item + Bug fix)
    class_counts_fold = np.bincount(y_np[train_idx], minlength=3)
    class_counts_fold = np.where(class_counts_fold == 0, 1, class_counts_fold)
    class_weights_fold = len(train_idx) / (3.0 * class_counts_fold)
    class_weights_tsr = torch.tensor(class_weights_fold, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tsr)

    train_ds = TensorDataset(X_train_tsr, y_train_tsr)
    val_ds = TensorDataset(X_val_tsr, y_val_tsr)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = BiGRUAttention(input_dim=10, num_classes=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)  # Mode strictly MIN for val_loss

    best_val_loss = float('inf')
    best_model_weights = None
    patience_counter = 0
    overfitting_gap_counter = 0

    train_losses, val_losses, train_accs, val_accs, val_f1s = [], [], [], [], []

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_b.size(0)
            _, preds = torch.max(out, 1)
            correct += (preds == y_b).sum().item()
            total += y_b.size(0)

        train_loss_epoch = train_loss / total
        train_acc_epoch = correct / total
        train_losses.append(train_loss_epoch)
        train_accs.append(train_acc_epoch)

        model.eval()
        val_loss, correct, total = 0, 0, 0
        all_val_preds, all_val_labels = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                out = model(X_b)
                loss = criterion(out, y_b)

                val_loss += loss.item() * X_b.size(0)
                _, preds = torch.max(out, 1)
                correct += (preds == y_b).sum().item()
                total += y_b.size(0)

                all_val_preds.extend(preds.cpu().numpy())
                all_val_labels.extend(y_b.cpu().numpy())

        val_loss_epoch = val_loss / total
        val_acc_epoch = correct / total
        val_losses.append(val_loss_epoch)
        val_accs.append(val_acc_epoch)

        val_f1_epoch = f1_score(all_val_labels, all_val_preds, average='macro', zero_division=0)
        val_f1s.append(val_f1_epoch)

        # Overfitting gap warning rule
        gap = abs(train_loss_epoch - val_loss_epoch)
        if gap > 0.15:
            overfitting_gap_counter += 1
        else:
            overfitting_gap_counter = 0

        if overfitting_gap_counter >= 5:
            print(f"WARNING: Overfitting detected — train/val loss gap has exceeded 0.15 for 5 consecutive epochs at fold {fold+1} epoch {epoch+1}.")

        scheduler.step(val_loss_epoch)

        # Early Stopping correctly tuned on val_loss
        if val_loss_epoch < best_val_loss:
            best_val_loss = val_loss_epoch
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(best_model_weights, os.path.join(base_dir, "models", "checkpoints", f"fold_{fold+1}_best.pt"))
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping triggered by val_loss at epoch {epoch+1}")
            model.load_state_dict(best_model_weights)
            break

    # Always load best weights post-training phase
    if best_model_weights:
        model.load_state_dict(best_model_weights)

    # 2 Subplots generation
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(train_losses, label='Train Loss')
    axes[0].plot(val_losses, label='Val Loss')
    axes[0].set_title(f'Fold {fold+1} Loss vs Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('CrossEntropyLoss')
    axes[0].legend()

    axes[1].plot(train_accs, label='Train Acc')
    axes[1].plot(val_accs, label='Val Acc')
    axes[1].set_title(f'Fold {fold+1} Acc vs Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(base_dir, "results", "training_curves", f"fold_{fold+1}_curves.png"))
    plt.close()

    # Best Model Inference (Metrics capture)
    model.eval()
    all_preds, all_labels = [], []
    all_probs = []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b = X_b.to(device)
            out = model(X_b)
            # raw out is already F.softmax probabilities because of model forward method
            probs = F.softmax(out, dim=1).cpu().numpy()
            all_probs.extend(probs)
            _, preds = torch.max(out, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_b.numpy())

    fold_val_preds_probs[fold+1] = all_probs
    fold_val_labels[fold+1] = all_labels

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Fold {fold+1}')
    plt.savefig(os.path.join(base_dir, "results", "metrics", f"fold_{fold+1}_confusion_matrix.png"))
    plt.close()

    report = classification_report(all_labels, all_preds, zero_division=0)
    with open(os.path.join(base_dir, "results", "metrics", f"final_classification_report_fold_{fold+1}.txt"), "w") as f:
        f.write(report)

    val_macro_f1_best = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    val_acc_best = sum([1 for a, p in zip(all_labels, all_preds) if a == p]) / len(all_labels)
    pr_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

    fold_f1_scores.append(val_macro_f1_best)
    cv_records.append({
        'fold': fold+1,
        'val_acc': val_acc_best,
        'macro_f1': val_macro_f1_best,
        'F1_class_0': pr_f1[0] if len(pr_f1) > 0 else 0,
        'F1_class_1': pr_f1[1] if len(pr_f1) > 1 else 0,
        'F1_class_2': pr_f1[2] if len(pr_f1) > 2 else 0
    })
print("Cross Validation Complete.")




--- Fold 1 Logging ---
Early stopping triggered by val_loss at epoch 21

--- Fold 2 Logging ---
Early stopping triggered by val_loss at epoch 31

--- Fold 3 Logging ---

--- Fold 4 Logging ---
Early stopping triggered by val_loss at epoch 45

--- Fold 5 Logging ---
Early stopping triggered by val_loss at epoch 30
Cross Validation Complete.


In [14]:
# Paste and run this right now
print("=== Per-Fold Results ===")
for i, rec in enumerate(cv_records):
    print(f"Fold {rec['fold']}: Acc={rec['val_acc']:.4f} | Macro-F1={rec['macro_f1']:.4f} | F1-class0={rec['F1_class_0']:.4f} | F1-class1={rec['F1_class_1']:.4f} | F1-class2={rec['F1_class_2']:.4f}")

print(f"\nBest fold by Macro-F1: Fold {np.argmax(fold_f1_scores)+1} → {max(fold_f1_scores):.4f}")
print(f"Mean Macro-F1 across folds: {np.mean(fold_f1_scores):.4f} ± {np.std(fold_f1_scores):.4f}")
print(f"Mean Accuracy across folds: {np.mean([r['val_acc'] for r in cv_records]):.4f} ± {np.std([r['val_acc'] for r in cv_records]):.4f}")

=== Per-Fold Results ===
Fold 1: Acc=0.8467 | Macro-F1=0.8495 | F1-class0=0.8213 | F1-class1=0.8508 | F1-class2=0.8765
Fold 2: Acc=0.9085 | Macro-F1=0.9144 | F1-class0=0.8617 | F1-class1=0.9156 | F1-class2=0.9660
Fold 3: Acc=0.9451 | Macro-F1=0.9465 | F1-class0=0.9160 | F1-class1=0.9510 | F1-class2=0.9726
Fold 4: Acc=0.8970 | Macro-F1=0.9031 | F1-class0=0.8446 | F1-class1=0.9057 | F1-class2=0.9589
Fold 5: Acc=0.9106 | Macro-F1=0.9088 | F1-class0=0.8934 | F1-class1=0.9179 | F1-class2=0.9150

Best fold by Macro-F1: Fold 3 → 0.9465
Mean Macro-F1 across folds: 0.9045 ± 0.0313
Mean Accuracy across folds: 0.9016 ± 0.0318


### Step 11 — Results, ROC Curves, & Summary saving

In [15]:
best_fold = np.argmax(fold_f1_scores) + 1
best_fold_f1 = fold_f1_scores[best_fold - 1]
print(f"\nSelected fold {best_fold} as final model with Macro-F1: {best_fold_f1:.4f}")

try:
    shutil.copy(
        os.path.join(base_dir, "models", "checkpoints", f"fold_{best_fold}_best.pt"),
        os.path.join(base_dir, "models", "final", "fatigue_model_final.pt")
    )
except Exception as e:
    print(e)

# cv_summary saving
cv_df = pd.DataFrame(cv_records).set_index('fold').T
cv_df['mean'] = cv_df.mean(axis=1)
cv_df['std'] = cv_df.std(axis=1)
cv_df.to_csv(os.path.join(base_dir, "results", "metrics", "cv_summary.csv"))
print("\n--- Cross Validation Summary ---")
print(cv_df)

# ROC Curves one-vs-rest
best_probs = np.array(fold_val_preds_probs[best_fold])
best_labels = label_binarize(fold_val_labels[best_fold], classes=[0,1,2])

plt.figure(figsize=(8,6))
colors = ['green', 'orange', 'red']
names = ["Non-Fatigued", "Mild Fatigue", "High Fatigue / Burnout"]

if best_labels.shape[1] == 3:
    for i in range(3):
        fpr, tpr, _ = roc_curve(best_labels[:, i], best_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[i], label=f'{names[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.title(f'One-vs-Rest ROC Curve (Best Fold: {best_fold})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.savefig(os.path.join(base_dir, "results", "metrics", "roc_curves.png"))
plt.close()




Selected fold 3 as final model with Macro-F1: 0.9465

--- Cross Validation Summary ---
fold               1         2         3         4         5      mean  \
val_acc     0.846682  0.908467  0.945080  0.897025  0.910550  0.901561   
macro_f1    0.849538  0.914419  0.946530  0.903062  0.908790  0.904468   
F1_class_0  0.821293  0.861660  0.915966  0.844622  0.893443  0.867397   
F1_class_1  0.850780  0.915612  0.951020  0.905660  0.917895  0.908193   
F1_class_2  0.876543  0.965986  0.972603  0.958904  0.915033  0.937814   

fold             std  
val_acc     0.031795  
macro_f1    0.031325  
F1_class_0  0.033807  
F1_class_1  0.032515  
F1_class_2  0.036666  


### Step 12 — Alpha Sensitivity Analysis

In [16]:
def run_alpha_sensitivity():
    print("\nRunning Alpha Sensitivity Setup (this will do 3-Fold 30 epoch fast validations)")
    alphas = [0.3, 0.4, 0.5, 0.6, 0.7]
    alpha_macro_f1_list = []

    # Utilizing raw arrays we retained explicitly for these kinds of adjustments
    for a in alphas:
        # Step A: Recompute A_final across full dataset temporarily locally
        A_fin = a * A_emotion + (1 - a) * A_acoustic
        F_sc = beta * (1 - A_fin) + (1 - beta) * emotion_probs[:, 1]

        # Step B: Regroup k-means locally
        km = KMeans(n_clusters=3, random_state=SEED, n_init=10)
        l_raw = km.fit_predict(F_sc.reshape(-1, 1))
        o = np.argsort(km.cluster_centers_.sum(axis=1))
        m = {o[0]: 0, o[1]: 1, o[2]: 2}
        temp_labels = np.array([m[l] for l in l_raw])
        features_df['temp_L_sens'] = temp_labels

        # Step C: Generate Alpha labels grouping array
        win_labels = []
        for speaker in features_df['speaker_id'].unique():
            spk_df = features_df[features_df['speaker_id'] == speaker].sort_values('file')
            if len(spk_df) < W: continue
            for i in range(0, len(spk_df) - W + 1, S):
                win = spk_df.iloc[i:i+W]
                maj = np.bincount(win['temp_L_sens'].values.astype(int)).argmax()
                win_labels.append(maj)

        y_alpha_np = np.array(win_labels)
        if len(y_alpha_np) < 3:
            alpha_macro_f1_list.append(0)
            continue

        skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
        fold_f1 = []

        # Fast 3 Fold Process
        for train_idx, val_idx in skf3.split(X_raw_np, y_alpha_np):
            train_raw = X_raw_np[train_idx]
            val_raw = X_raw_np[val_idx]
            train_emo = X_emo_np[train_idx]
            val_emo = X_emo_np[val_idx]

            sf = MinMaxScaler()
            sf.fit(train_raw.reshape(-1, 6))

            t_scaled = sf.transform(train_raw.reshape(-1, 6)).reshape(train_raw.shape)
            v_scaled = sf.transform(val_raw.reshape(-1, 6)).reshape(val_raw.shape)

            def qb(ac_scaled, emo):
                A_ac = 0.3*ac_scaled[:,:,0] + 0.25*ac_scaled[:,:,1] + 0.15*ac_scaled[:,:,2] + 0.15*ac_scaled[:,:,5] + 0.10*ac_scaled[:,:,3] + 0.05*ac_scaled[:,:,4]
                A_em = 0.8*emo[:,:,0] + 0.2*emo[:,:,1] + 0.4*emo[:,:,2]
                A_fn = a * A_em + (1 - a) * A_ac
                F_sn = beta * (1 - A_fn) + (1 - beta) * emo[:,:,1]

                out = np.zeros((ac_scaled.shape[0], W, 10))
                out[:,:,0:6] = ac_scaled; out[:,:,6] = A_em; out[:,:,7] = A_fn; out[:,:,8] = F_sn
                for idx in range(ac_scaled.shape[0]):
                    s, _, _, _, _ = linregress(range(W), A_fn[idx])
                    out[idx, :, 9] = s if not np.isnan(s) else 0
                return torch.tensor(out, dtype=torch.float32)

            X_t = qb(t_scaled, train_emo)
            X_v = qb(v_scaled, val_emo)
            Y_t = torch.tensor(y_alpha_np[train_idx], dtype=torch.long)
            Y_v = torch.tensor(y_alpha_np[val_idx], dtype=torch.long)

            md = BiGRUAttention(input_dim=10, num_classes=3).to(device)
            opt = torch.optim.AdamW(md.parameters(), lr=1e-3)
            crit = nn.CrossEntropyLoss()

            train_ldr = DataLoader(TensorDataset(X_t, Y_t), batch_size=64, shuffle=True)

            best_l = float('inf')
            best_wt = None
            pat_c = 0
            for ep in range(30):
                md.train()
                for xb, yb in train_ldr:
                    xb, yb = xb.to(device), yb.to(device)
                    opt.zero_grad()
                    crit(md(xb), yb).backward()
                    opt.step()

                md.eval()
                with torch.no_grad():
                    vl = crit(md(X_v.to(device)), Y_v.to(device)).item()
                if vl < best_l:
                    best_l = vl
                    best_wt = copy.deepcopy(md.state_dict())
                    pat_c = 0
                else:
                    pat_c += 1
                    if pat_c >= 5: break

            if best_wt:
                md.load_state_dict(best_wt)
            md.eval()
            with torch.no_grad():
                _, pr = torch.max(md(X_v.to(device)), 1)
                f1_val = f1_score(Y_v.numpy(), pr.cpu().numpy(), average='macro', zero_division=0)
                fold_f1.append(f1_val)

        mean_f1 = np.mean(fold_f1)
        alpha_macro_f1_list.append(mean_f1)
        print(f"Alpha {a} -> Mean Macro F1: {mean_f1:.4f}")

    plt.figure()
    plt.plot(alphas, alpha_macro_f1_list, marker='o', linestyle='-')
    plt.title('Alpha Sensitivity Analysis on Test Accuracy (Macro-F1)')
    plt.xlabel('Alpha')
    plt.ylabel('Mean Macro-F1 across 3-Folds')
    plt.grid()
    plt.savefig(os.path.join(base_dir, "results", "metrics", "alpha_sensitivity.png"))
    plt.close()

try:
    run_alpha_sensitivity()
except Exception as e:
    print(f"Alpha sensitivity experienced an error: {e}")




Running Alpha Sensitivity Setup (this will do 3-Fold 30 epoch fast validations)
Alpha 0.3 -> Mean Macro F1: 0.5247
Alpha 0.4 -> Mean Macro F1: 0.5066
Alpha 0.5 -> Mean Macro F1: 0.4256
Alpha 0.6 -> Mean Macro F1: 0.4092
Alpha 0.7 -> Mean Macro F1: 0.6367


### Step 13 — Inference Function

In [17]:
def predict_fatigue(audio_paths: list, model, emotion_model, scaler, W=10) -> dict:
    if len(audio_paths) != W:
        raise ValueError(f"Expected strictly W={W} audio paths forming a session window.")

    seq_ac = []
    seq_emo = []

    for p in audio_paths:
        feats = extract_acoustic_features(p)
        p_pos, p_neg, p_neu = get_emotion_probs(p, emotion_model, device)
        seq_ac.append(feats)
        seq_emo.append([p_pos, p_neg, p_neu])

    seq_ac = np.array(seq_ac) # (W, 6)
    seq_emo = np.array(seq_emo) # (W, 3)

    scaled_ac = scaler.transform(seq_ac)

    A_ac = 0.3*scaled_ac[:,0] + 0.25*scaled_ac[:,1] + 0.15*scaled_ac[:,2] + 0.15*scaled_ac[:,5] + 0.10*scaled_ac[:,3] + 0.05*scaled_ac[:,4]
    A_em = 0.8*seq_emo[:,0] + 0.2*seq_emo[:,1] + 0.4*seq_emo[:,2]
    A_fin = alpha * A_em + (1 - alpha) * A_ac
    F_sc = beta * (1 - A_fin) + (1 - beta) * seq_emo[:,1]

    slope, _, _, _, _ = linregress(range(W), A_fin)
    if np.isnan(slope): slope = 0

    X_infer = np.zeros((1, W, 10))
    X_infer[0, :, 0:6] = scaled_ac
    X_infer[0, :, 6] = A_em
    X_infer[0, :, 7] = A_fin
    X_infer[0, :, 8] = F_sc
    X_infer[0, :, 9] = slope

    X_tsr = torch.tensor(X_infer, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(X_tsr)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        label_pred = int(np.argmax(probs))

    m = {0: "Non-Fatigued", 1: "Mild Fatigue", 2: "High Fatigue / Burnout"}

    return {
        "fatigue_score": float(np.mean(F_sc) + gamma * max(0, -slope)),
        "label": label_pred,
        "label_str": m[label_pred],
        "class_probs": probs.tolist(),
        "arousal_trend": float(slope)
    }



In [18]:
# Check what your CREMA-D files look like in features_df
print(features_df[features_df['dataset']=='CREMA-D']['file'].head(5).values)

# Then check the actual folder structure
import glob
sample = glob.glob(os.path.join(raw_data_dir, '**', '*.wav'), recursive=True)[:3]
print(sample)

['1080_DFA_NEU_XX.wav' '1079_TIE_FEA_XX.wav' '1080_DFA_DIS_XX.wav'
 '1079_WSI_ANG_XX.wav' '1080_DFA_ANG_XX.wav']
['/content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_DFA_NEU_XX.wav', '/content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1079_TIE_FEA_XX.wav', '/content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_DFA_DIS_XX.wav']


In [19]:
# Load the final saved model
final_model = BiGRUAttention(input_dim=10, num_classes=3).to(device)
final_model.load_state_dict(
    torch.load(os.path.join(base_dir, 'models', 'final', 'fatigue_model_final.pt'),
               map_location=device)
)
final_model.eval()

# Load the saved scaler
with open(os.path.join(npy_dir, 'feature_scaler.pkl'), 'rb') as f:
    saved_scaler = pickle.load(f)

# Pick 10 files from the same speaker
test_speaker_files = features_df[
    features_df['speaker_id'] == '1080'
].sort_values('file')['file'].values[:10]

# Build full paths — CREMA-D is inside AudioWAV subfolder
test_paths = [
    os.path.join(raw_data_dir, 'CREMA-D', 'AudioWAV', fname)
    for fname in test_speaker_files
]

print("Test files:")
for p in test_paths:
    print(" ", p)

# Run inference
result = predict_fatigue(
    audio_paths=test_paths,
    model=final_model,
    emotion_model=emotion_model,
    scaler=saved_scaler,
    W=10
)

print("\n=== Fatigue Inference Result ===")
print(f"Fatigue Score   : {result['fatigue_score']:.4f}  (0=no fatigue, 1=severe burnout)")
print(f"Label           : {result['label']} — {result['label_str']}")
print(f"Class Probs     : Non-Fatigued={result['class_probs'][0]:.3f} | Mild={result['class_probs'][1]:.3f} | High={result['class_probs'][2]:.3f}")
print(f"Arousal Trend   : {result['arousal_trend']:.4f} ({'⬇ Declining — fatigue accumulating' if result['arousal_trend'] < 0 else '⬆ Stable/Rising — no fatigue trend'})")

Test files:
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_DFA_ANG_XX.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_DFA_DIS_XX.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_DFA_FEA_XX.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_DFA_HAP_XX.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_DFA_NEU_XX.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_DFA_SAD_XX.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_IEO_ANG_HI.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_IEO_ANG_LO.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_IEO_ANG_MD.wav
  /content/drive/MyDrive/Fatigue_Burnout_detection/data/raw/CREMA-D/AudioWAV/1080_IEO_DIS_HI.wav

=== Fatigue Infer